Sumário do modelo treinado

---

Bibliotecas

In [2]:
import torch
import torch.nn as nn

from torchinfo import summary

---

Definição do modelo

In [3]:
class ECGReconstructor(nn.Module):

    def __init__(self, latentDim, hiddenDim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(12, hiddenDim, 5, stride=2, padding=2),
            nn.BatchNorm1d(hiddenDim),
            nn.ReLU(),

            nn.Conv1d(hiddenDim, latentDim, 5, stride=2, padding=2),
            nn.BatchNorm1d(latentDim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(latentDim, hiddenDim, 4, stride=2, padding=1),
            nn.ReLU(),

            nn.ConvTranspose1d(hiddenDim, 12, 4, stride=2, padding=1),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        z = self.encoder(x)
        out = self.decoder(z)
        return out.permute(0, 2, 1)

---

Sumário do modelo

In [4]:
model = ECGReconstructor(
    latentDim = 128,
    hiddenDim = 32
)

In [15]:
summary(model, input_size = (32, 2896, 12))

Layer (type:depth-idx)                   Output Shape              Param #
ECGReconstructor                         [32, 2896, 12]            --
├─Sequential: 1-1                        [32, 128, 724]            --
│    └─Conv1d: 2-1                       [32, 32, 1448]            1,952
│    └─BatchNorm1d: 2-2                  [32, 32, 1448]            64
│    └─ReLU: 2-3                         [32, 32, 1448]            --
│    └─Conv1d: 2-4                       [32, 128, 724]            20,608
│    └─BatchNorm1d: 2-5                  [32, 128, 724]            256
│    └─ReLU: 2-6                         [32, 128, 724]            --
├─Sequential: 1-2                        [32, 12, 2896]            --
│    └─ConvTranspose1d: 2-7              [32, 32, 1448]            16,416
│    └─ReLU: 2-8                         [32, 32, 1448]            --
│    └─ConvTranspose1d: 2-9              [32, 12, 2896]            1,548
Total params: 40,844
Trainable params: 40,844
Non-trainable params: 0
